### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "weighted" / "RandomForest_thres0.5_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.12,,,,,
1,0,cf_1,,,,,,,16.9905,,,0.125,1,False,0.4789,0.4679
2,0,cf_2,,,,,,,17.107,5,,0.207,2,False,0.4789,0.3557
3,0,cf_3,,,,7,,,18.9745,7,,0.2508,3,False,0.4789,0.4179
4,0,cf_4,,,5,,,,18.9619,2,,0.1206,3,False,0.4789,0.3592
5,0,cf_5,2,,,,,,18.9619,3,,0.1801,3,False,0.4789,0.3562
6,0,cf_6,3,,,,,,18.9619,3,,0.1176,3,False,0.4789,0.3536
7,0,cf_7,,,6,,,,18.9565,2,,0.1626,3,False,0.4789,0.361
8,0,cf_8,,,,6,,,18.2558,7,,0.2541,3,False,0.4789,0.4061
9,0,cf_9,,,6,,,,18.2499,4,,0.2426,3,False,0.4789,0.3614


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.12,,,,,
9,0,cf_1,,,,,,,16.9905,,,0.125,1,False,0.4789,0.4679
1,1,xin,3,4,1,2,3,0,22.3757,0,2.14,,,,,
10,1,cf_1,,,,,2,,22.3757,,,0.0671,2,False,0.5551,0.4949
2,2,xin,5,3,1,1,4,0,22.694,7,4.31,,,,,
11,2,cf_1,,1,,4,,,22.68,,,0.2962,3,False,0.5325,0.4645
3,3,xin,3,3,6,6,2,0,24.3418,1,3.13,,,,,
12,3,cf_1,,,,,,,23.4856,,,0.0256,1,False,0.4623,0.4549
4,4,xin,5,4,2,7,1,0,21.2585,3,3.38,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,False,0.2538,0.2538


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.12,,,,,
9,0,cf_1,,,,,,,16.9905,,,0.125,1,False,0.4789,0.4679
1,1,xin,3,4,1,2,3,0,22.3757,0,2.14,,,,,
10,1,cf_1,,,,,2,,22.3757,,,0.0671,2,False,0.5551,0.4949
2,2,xin,5,3,1,1,4,0,22.694,7,4.31,,,,,
11,2,cf_1,,1,,4,,,22.68,,,0.2962,3,False,0.5325,0.4645
3,3,xin,3,3,6,6,2,0,24.3418,1,3.13,,,,,
12,3,cf_1,,,,,,,23.4856,,,0.0256,1,False,0.4623,0.4549
4,4,xin,5,4,2,7,1,0,21.2585,3,3.38,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,False,0.2538,0.2538


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.4679
Valid: False
Changes:
  bmi: 18.9866 → 16.9905


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.4949
Valid: False
Changes:
  slprl: 3 → 2


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance:
  etfruit: 5
  eatveg: 3
  cgtsmok: 1
  alcfreq: 1
  slprl: 4
  paccnois: 0
  bmi: 22.694
  dosprt: 7


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.4645
Valid: False
Changes:
  eatveg: 3 → 1
  alcfreq: 1 → 4
  bmi: 22.694 → 22.68


=== Query index '3' ===
Task / Target: hltprhc
Query instance 